# Этап 2: Сбор HTML-статей

**Источники HTML (в порядке приоритета):**
1. `export.arxiv.org/html/` — выделенный сервер arXiv для программного доступа
2. `ar5iv.labs.arxiv.org/html/` — зеркало HTML-версий (fallback)

**Rate limiting (по рекомендации arXiv):**
- Burst: до 4 запросов в секунду
- После burst: пауза 1 секунда
- Источник: https://info.arxiv.org/help/bulk_data.html

**Примечание:** HTML-версии НЕ доступны через S3/Kaggle bulk access.
Они генерируются LaTeXML и доступны только через HTTP.
~8-10% статей не имеют HTML-версии (LaTeXML не справился с конвертацией).

In [1]:
import pandas as pd
import httpx
import asyncio
import random
from pathlib import Path
from tqdm.notebook import tqdm

import pyarrow as pa
import pyarrow.parquet as pq
import math

In [2]:
# === КОНФИГУРАЦИЯ ===
DATA_PATH = Path('../../data/raw/html/')
DATA_PATH.mkdir(parents=True, exist_ok=True)
METADATA_PATH = Path('../../data/metadata/arxiv_NLP_2021_2026_metadata.csv')

# Rate limiting по рекомендации arXiv
BURST_SIZE = 4              # Запросов в одном burst
BURST_PAUSE = 1.0           # Пауза между burst'ами (секунды)
MAX_RETRIES = 3             # Попыток на статью при rate limit
MIN_CONTENT_SIZE = 10_000   # Минимальный размер HTML (байт), меньше = заглушка

# Честный User-Agent (arXiv просит идентифицировать ботов)
HEADERS = {
    "User-Agent": "ArXivRAGProject/1.0 (academic capstone; https://github.com/ConstDemi/ragarxiv)",
    "Accept": "text/html",
}

In [3]:
# === ЗАГРУЗКА ДАННЫХ ===
df = pd.read_csv(METADATA_PATH, usecols=['arxiv_id', 'title', 'published', 'html_url'])
df['published'] = pd.to_datetime(df['published'])
df_2025 = df[df['published'].dt.year == 2025].copy()

print(f'Статей за 2025 год: {df_2025.shape[0]}')

# Проверка уже скачанных
downloaded_ids = {f.stem for f in DATA_PATH.glob('*.html')}
ids_to_download = df_2025[~df_2025['arxiv_id'].isin(downloaded_ids)]

print(f'Уже скачано: {len(downloaded_ids)}')
print(f'Осталось: {ids_to_download.shape[0]}')

Статей за 2025 год: 856
Уже скачано: 825
Осталось: 31


In [4]:
# === ЛОГИКА СКАЧИВАНИЯ ===

failed_downloads = []


def get_urls(arxiv_id: str) -> list[str]:
    """
    URL'ы для попыток скачивания.
    Приоритет: export.arxiv.org (рекомендуемый для ботов) -> ar5iv (fallback).
    """
    return [
        f"https://export.arxiv.org/html/{arxiv_id}",
        f"https://ar5iv.labs.arxiv.org/html/{arxiv_id}",
    ]


async def download_one(client: httpx.AsyncClient, arxiv_id: str) -> bool:
    """
    Скачивает одну статью. Пробует основной URL, при неудаче — fallback.
    Retry с exponential backoff при rate limit (429/503).
    Возвращает True при успехе.
    """
    urls = get_urls(arxiv_id)
    last_error = None

    for url in urls:
        for attempt in range(MAX_RETRIES):
            try:
                response = await client.get(url)

                if response.status_code == 200:
                    if len(response.text) >= MIN_CONTENT_SIZE:
                        filepath = DATA_PATH / f'{arxiv_id}.html'
                        with open(filepath, 'w', encoding='utf-8') as f:
                            f.write(response.text)
                        return True
                    else:
                        # Заглушка/пустая страница — пробуем другой URL
                        last_error = {"status": 200, "type": "SmallContent", "url": url}
                        break

                elif response.status_code == 404:
                    # Нет HTML-версии — пробуем другой URL
                    last_error = {"status": 404, "type": "NotFound", "url": url}
                    break

                elif response.status_code in (429, 503, 406):
                    # Rate limit — ждём и повторяем
                    wait = (2 ** attempt) + random.uniform(0.5, 1.5)
                    last_error = {"status": response.status_code, "type": "RateLimit", "url": url}
                    await asyncio.sleep(wait)
                    continue

                else:
                    last_error = {"status": response.status_code, "type": "HTTPError", "url": url}
                    break

            except httpx.TimeoutException:
                last_error = {"status": None, "type": "Timeout", "url": url}
                await asyncio.sleep(2 ** attempt)
            except Exception as e:
                last_error = {"status": None, "type": "Exception", "url": url, "details": str(e)}
                break

    # Все попытки провалились
    failed_downloads.append({"arxiv_id": arxiv_id, **(last_error or {"type": "Unknown"})})
    return False

In [5]:
# === ОСНОВНОЙ ЦИКЛ: burst-скачивание по рекомендации arXiv ===

async def main():
    rows = list(ids_to_download['arxiv_id'])
    total = len(rows)

    if total == 0:
        print("Всё уже скачано!")
        return

    # Оценка времени: (total / BURST_SIZE) * (1 + BURST_PAUSE) секунд
    est_minutes = (total / BURST_SIZE) * (1 + BURST_PAUSE) / 60
    print(f"Скачиваем {total} статей")
    print(f"Режим: burst по {BURST_SIZE}, пауза {BURST_PAUSE}с")
    print(f"Сервер: export.arxiv.org (рекомендуемый для ботов)")
    print(f"Примерное время: ~{est_minutes:.0f} мин\n")

    async with httpx.AsyncClient(
        timeout=30.0,
        follow_redirects=True,
        headers=HEADERS,
        limits=httpx.Limits(max_connections=BURST_SIZE, max_keepalive_connections=BURST_SIZE)
    ) as client:

        pbar = tqdm(total=total, desc="Downloading")
        success_count = 0

        # Обрабатываем burst'ами
        for i in range(0, total, BURST_SIZE):
            burst = rows[i : i + BURST_SIZE]

            # Запускаем burst параллельно
            tasks = [download_one(client, arxiv_id) for arxiv_id in burst]
            results = await asyncio.gather(*tasks)

            success_count += sum(results)
            pbar.update(len(burst))

            # Пауза между burst'ами
            if i + BURST_SIZE < total:
                await asyncio.sleep(BURST_PAUSE)

        pbar.close()
        print(f"\nУспешно скачано: {success_count}/{total}")
        print(f"Ошибок: {len(failed_downloads)}")

await main()

Скачиваем 31 статей
Режим: burst по 4, пауза 1.0с
Сервер: export.arxiv.org (рекомендуемый для ботов)
Примерное время: ~0 мин



Downloading:   0%|          | 0/31 [00:00<?, ?it/s]


Успешно скачано: 0/31
Ошибок: 31


In [6]:
# === ИТОГОВАЯ СТАТИСТИКА ===
downloaded_ids = {f.stem for f in DATA_PATH.glob('*.html')}
total_2025 = df_2025.shape[0]
coverage = len(downloaded_ids) / total_2025 * 100

print(f'Итого скачано: {len(downloaded_ids)} / {total_2025} ({coverage:.1f}%)')

Итого скачано: 825 / 856 (96.4%)


In [7]:
# === АНАЛИЗ ОШИБОК ===
if failed_downloads:
    df_errors = pd.DataFrame(failed_downloads)

    print("Распределение по типу ошибки:")
    print(df_errors['type'].value_counts())
    print()

    if 'status' in df_errors.columns:
        print("Распределение по статус-коду:")
        print(df_errors['status'].value_counts())
        print()

    # Классификация
    n_no_html = len(df_errors[df_errors['status'] == 404])
    n_rate_limit = len(df_errors[df_errors['type'] == 'RateLimit'])
    n_other = len(df_errors) - n_no_html - n_rate_limit

    print(f'Нет HTML-версии (404): {n_no_html} — нормально, LaTeXML не конвертировал')
    print(f'Rate limit (429/503/406): {n_rate_limit} — можно перезапустить нотбук')
    print(f'Прочие ошибки: {n_other}')

    # Сохраняем отчёт
    df_errors.to_csv('../../data/metadata/download_errors.csv', index=False)
else:
    print("Ошибок не зафиксировано!")

Распределение по типу ошибки:
type
SmallContent    31
Name: count, dtype: int64

Распределение по статус-коду:
status
200    31
Name: count, dtype: int64

Нет HTML-версии (404): 0 — нормально, LaTeXML не конвертировал
Rate limit (429/503/406): 0 — можно перезапустить нотбук
Прочие ошибки: 31


---
# Конвертация HTML -> Parquet для дальнейшей обработки

In [8]:
RAW_HTML_DIR = Path('../../data/raw/html').resolve()
RAW_PARQUET_DIR = Path('../../data/raw/parquet').resolve()
RAW_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
SHARD_SIZE = 1000

html_files = list(RAW_HTML_DIR.glob('*.html'))
print(f'HTML файлов для конвертации: {len(html_files)}')

HTML файлов для конвертации: 825


In [9]:
schema = pa.schema([
    ('doc_id', pa.string()),
    ('html', pa.string())
])


def process_shards(files, shard_size, dest_dir):
    for i in tqdm(range(0, len(files), shard_size), desc='Processing Shards'):
        batch_files = files[i : i + shard_size]
        batch_data = []

        for f in batch_files:
            try:
                content = f.read_text(encoding='utf-8')
                batch_data.append({'doc_id': f.stem, 'html': content})
            except Exception as e:
                print(f'Ошибка в файле {f.name}: {e}')

        if batch_data:
            table = pa.Table.from_pylist(batch_data, schema=schema)
            shard_name = f'shard_{i // shard_size:04d}.parquet'
            pq.write_table(table, dest_dir / shard_name)


process_shards(html_files, SHARD_SIZE, RAW_PARQUET_DIR)

Processing Shards:   0%|          | 0/1 [00:00<?, ?it/s]